<a href="https://colab.research.google.com/github/ju-log/Memory/blob/Momentum-3rd-project/%EA%B1%B4%EA%B0%95%EB%B3%B4%ED%97%98%EC%8B%AC%EC%82%AC%ED%8F%89%EA%B0%80%EC%9B%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import math
import time
from google.colab import files

# 본인의 Decoding 인증키 입력
SERVICE_KEY = "여기에_인증키_입력"

def xml_to_rows(content):
    root = ET.fromstring(content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")

    if result_code not in (None, "00"):
        raise Exception(f"API 오류: {result_code} / {result_msg}")

    rows = []

    for item in root.findall(".//item"):
        row = {}

        for child in item:
            row[child.tag] = child.text

        rows.append(row)

    total_count = int(root.findtext(".//totalCount") or len(rows))

    return rows, total_count

print("준비 완료")

준비 완료


In [ ]:
import requests
import pandas as pd
import math
import time
from google.colab import files

# 1번: 비급여항목병원목록상세
URL = "https://apis.data.go.kr/B551182/nonPaymentDamtInfoService/getNonPaymentItemHospDtlList"

ROWS = 100
all_rows = []

params = {
    "ServiceKey": 여기에 인증키 입력,
    "pageNo": 1,
    "numOfRows": ROWS,
    "ykiho": "",
    "clCd": "",
    "sidoCd": "110000",   # 서울
    "sgguCd": "",
    "yadmNm": ""
}

# 1페이지 먼저 조회
response = requests.get(URL, params=params, timeout=90)
response.raise_for_status()

rows, total_count = xml_to_rows(response.content)
all_rows.extend(rows)

total_pages = math.ceil(total_count / ROWS)

print("전체 건수:", total_count)
print("전체 페이지:", total_pages)
print(f"1/{total_pages} 페이지 완료")


# 2페이지부터 마지막 페이지까지 수집
for page in range(2, total_pages + 1):

    params["pageNo"] = page
    success = False

    # 오류 발생 시 최대 5번 재시도
    for attempt in range(5):
        try:
            response = requests.get(
                URL,
                params=params,
                timeout=90
            )

            response.raise_for_status()

            rows, _ = xml_to_rows(response.content)
            all_rows.extend(rows)

            success = True
            break

        except requests.exceptions.RequestException:
            print(
                f"{page}페이지 오류 → "
                f"{attempt + 1}/5 재시도"
            )
            time.sleep(5 * (attempt + 1))

    if not success:
        print(f"{page}페이지에서 반복 실패하여 중단합니다.")
        break

    if page % 20 == 0 or page == total_pages:
        print(f"{page}/{total_pages} 페이지 완료")

    # 서버에 너무 빠르게 요청하지 않기
    time.sleep(0.3)


# CSV로 저장
df_price = pd.DataFrame(all_rows)

print()
print("수집된 행 수:", len(df_price))

filename = "01_서울_병원별_비급여가격.csv"

df_price.to_csv(
    filename,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", filename)

files.download(filename)

전체 건수: 49198
전체 페이지: 492
1/492 페이지 완료
20/492 페이지 완료
40/492 페이지 완료
60/492 페이지 완료
80/492 페이지 완료
94페이지 오류 → 1/5 재시도
100/492 페이지 완료
120/492 페이지 완료
140/492 페이지 완료
160/492 페이지 완료
180/492 페이지 완료
200/492 페이지 완료
220/492 페이지 완료
240/492 페이지 완료
260/492 페이지 완료
280/492 페이지 완료
300/492 페이지 완료
320/492 페이지 완료
340/492 페이지 완료
360/492 페이지 완료
380/492 페이지 완료
400/492 페이지 완료
420/492 페이지 완료
437페이지 오류 → 1/5 재시도
440/492 페이지 완료
460/492 페이지 완료
480/492 페이지 완료
492/492 페이지 완료

수집된 행 수: 49198
저장 완료: 01_서울_병원별_비급여가격.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 3번: 비급여항목코드조회(16.3월 이후)

URL = "https://apis.data.go.kr/B551182/nonPaymentDamtInfoService/getNonPaymentItemCodeList2"

ROWS = 100
page = 1
all_rows = []

# 1페이지 조회
params = {
    "ServiceKey": 여기에 인증키 입력,
    "pageNo": page,
    "numOfRows": ROWS
}

response = requests.get(URL, params=params, timeout=60)
response.raise_for_status()

rows, total_count = xml_to_rows(response.content)
all_rows.extend(rows)

total_pages = math.ceil(total_count / ROWS)

print("전체 건수:", total_count)
print("전체 페이지:", total_pages)
print(f"1/{total_pages} 페이지 완료")


# 나머지 페이지 자동 수집
for page in range(2, total_pages + 1):

    params["pageNo"] = page

    response = requests.get(URL, params=params, timeout=60)
    response.raise_for_status()

    rows, _ = xml_to_rows(response.content)
    all_rows.extend(rows)

    print(f"{page}/{total_pages} 페이지 완료")

    time.sleep(0.05)


# 데이터프레임 생성
df_code = pd.DataFrame(all_rows)

print()
print("수집 완료!")
print("최종 행 수:", len(df_code))

display(df_code.head())


# CSV 저장
filename = "02_비급여항목코드.csv"

df_code.to_csv(
    filename,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", filename)


# 내 컴퓨터로 다운로드
files.download(filename)

전체 건수: 875
전체 페이지: 9
1/9 페이지 완료
2/9 페이지 완료
3/9 페이지 완료
4/9 페이지 완료
5/9 페이지 완료
6/9 페이지 완료
7/9 페이지 완료
8/9 페이지 완료
9/9 페이지 완료

수집 완료!
최종 행 수: 875


,adtEndDd,adtFrDd,npayCd,npayDtlDivCd,npayDtlDivCdNm,npayKorNm,npayMdivCd,npayMdivCdNm,npaySdivCd,npaySdivCdNm,cmmtTxt
0,99991231,20241126,1010A,1010A,상급병실료,상급병실료,1010A,상급병실료,1010A,상급병실료,NaN
1,99991231,20241126,ABZ010001,1010A010,1인실,상급병실료/1인실,1010A,상급병실료,1010A010,1인실,NaN
2,99991231,20241126,ABZ020001,1010A020,2인실,상급병실료/2인실,1010A,상급병실료,1010A020,2인실,NaN
3,99991231,20241126,ABZ030001,1010A030,3인실,상급병실료/3인실,1010A,상급병실료,1010A030,3인실,NaN
4,99991231,20241126,ABZ040001,1010A040,4인실,상급병실료/4인실,1010A,상급병실료,1010A040,4인실,NaN


저장 완료: 02_비급여항목코드.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

# 원본 CSV 읽기
df = pd.read_csv(
    "01_서울_병원별_비급여가격.csv",
    encoding="utf-8-sig"
)

# 앞 300행만 샘플로 저장
sample = df.head(300)

sample_filename = "sample_서울_병원별_비급여가격.csv"

sample.to_csv(
    sample_filename,
    index=False,
    encoding="utf-8-sig"
)

print("샘플 행 수:", len(sample))

files.download(sample_filename)

샘플 행 수: 300


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>